# AIC25 — Fast Tier-1 Path (Single-Camera JSONs + Tracklet Repair)

Generates single-camera tracking JSONs for a few Warehouse_016 cameras **without training**, then runs the `tracklet_repair` ablation → **Tier-1 comparison table**.

**Fixes baked in (learned the hard way):**
- OSNet weights pulled from the **HuggingFace mirror** (the original Google-Drive link is dead).
- `EmbedFeature/` written to **local disk** (Drive small-file writes are ~100× too slow).
- Detection re-run **fresh + frame-capped** each time (avoids stale/truncated cached detections that hang).

Branch: **`main`**. Run top-to-bottom on a **T4 GPU**.

---
## Step 0 — Environment + Drive

In [ ]:
import os, sys, shutil
ON_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
if ON_COLAB:
    REPO='/content/repo'; PY='python'; DRIVE='/content/drive/MyDrive/AIC25'
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'): drive.mount('/content/drive')
    else: print('Drive already mounted.')
    for d in ['models','outputs/Detection','outputs/Tracking']:
        os.makedirs(f'{DRIVE}/{d}', exist_ok=True)
    print('Colab | Drive:', DRIVE)
else:
    REPO='/home/seco/deepLearning/Single-Camera-Tracking-Consistency'; PY=f'{REPO}/.venv/bin/python'; DRIVE=None
    os.chdir(REPO); print('Local')
print('REPO:', REPO)

---
## Step 1 — Clone + checkout repo branch + install

In [ ]:
if ON_COLAB:
    import subprocess as _sp
    BRANCH = 'main'
    GIT_URL = 'https://github.com/Hithesh18/Single-Camera-Tracking-Consistency.git'

    def _run(cmd, check=True):
        result = _sp.run(cmd, capture_output=True, text=True)
        if check and result.returncode != 0:
            detail = (result.stderr or result.stdout).strip()
            raise RuntimeError(f"{' '.join(cmd)} failed:\n{detail}")
        return result

    if not os.path.exists(f'{REPO}/.git'):
        if os.path.exists(REPO): shutil.rmtree(REPO)
        _run(['git', 'clone', '--depth', '1', '--branch', BRANCH, GIT_URL, REPO])
    else:
        _run(['git', '-C', REPO, 'fetch', 'origin', BRANCH])
        _run(['git', '-C', REPO, 'checkout', BRANCH])
        _run(['git', '-C', REPO, 'pull', '--ff-only', 'origin', BRANCH])

    os.chdir(REPO)
    if not os.path.isdir(f'{REPO}/tracklet_repair'):
        raise RuntimeError('tracklet_repair missing after checkout of main')
    print('On main ✓')
    setup_marker = f'{REPO}/.colab_deps_ok_tier1'
    if os.path.exists(setup_marker):
        print('Dependencies already installed for this runtime.')
    else:
        for _p in ['thop','loguru','lap','motmetrics','filterpy','easydict','yacs','termcolor',
                   'prettytable','tabulate','ninja','cython_bbox','pycocotools','huggingface_hub']:
            _sp.run(['pip','install','-q',_p], capture_output=True, text=True)
        if _sp.run(['pip','install','-q','faiss-gpu'], capture_output=True).returncode != 0:
            _sp.run(['pip','install','-q','faiss-cpu'], capture_output=True)
        os.chdir(f'{REPO}/BoT-SORT');         os.system('python setup.py develop --quiet 2>/dev/null')
        os.chdir(f'{REPO}/deep-person-reid'); os.system('python setup.py develop --quiet 2>/dev/null')
        os.chdir(REPO); os.system('pip install -q -r tracking/requirements.txt 2>/dev/null')
        open(setup_marker, 'w').write('ok\n')
    _v = _sp.run('python -c "from yolox.exp import get_exp; from tracker.bot_sort import BoTSORT; print(\'ALL OK\')"',
                 shell=True, capture_output=True, text=True)
    print('✓ Dependencies verified.' if 'ALL OK' in _v.stdout else '✗ FAILED: '+_v.stderr.strip()[-400:])
else:
    print('Local: skip.')

---
## Step 2 — GPU check (required)

In [ ]:
import subprocess
r = subprocess.run([PY,'-c','import torch; print("CUDA:", torch.cuda.is_available())'], capture_output=True, text=True)
print(r.stdout.strip())
if ON_COLAB and 'CUDA: False' in r.stdout:
    raise RuntimeError('NO GPU — Runtime → Change runtime type → T4 GPU, Restart, re-run.')

---
## Step 3 — Models (OSNet from HF mirror + ByteTrack fallback)
The original OSNet Google-Drive link is **dead**, so we use the author's HuggingFace mirror `kaiyangzhou/osnet` and save it under the filename the code expects. No AIC25 detector needed (ByteTrack fallback).

In [ ]:
if ON_COLAB:
    from huggingface_hub import hf_hub_download
    M = f'{DRIVE}/models'; os.makedirs(M, exist_ok=True)

    # --- OSNet (reliable HF mirror) ---
    osnet_local = f'{REPO}/deep-person-reid/checkpoints/osnet_ms_m_c.pth.tar'
    osnet_drive = f'{M}/osnet_ms_m_c.pth.tar'
    os.makedirs(os.path.dirname(osnet_local), exist_ok=True)
    if os.path.exists(osnet_local):
        print('OSNet: already local')
    elif os.path.exists(osnet_drive):
        shutil.copy(osnet_drive, osnet_local); print('OSNet: from Drive')
    else:
        fn = 'osnet_x1_0_msmt17_combineall_256x128_amsgrad_ep150_stp60_lr0.0015_b64_fb10_softmax_labelsmooth_flip_jitter.pth'
        src = hf_hub_download(repo_id='kaiyangzhou/osnet', filename=fn)
        shutil.copy(src, osnet_local); shutil.copy(osnet_local, osnet_drive)
        print('OSNet: downloaded from HF mirror →', os.path.getsize(osnet_local), 'bytes')

    # --- ByteTrack fallback detector ---
    bt_local = f'{REPO}/BoT-SORT/pretrained/bytetrack_x_mot17.pth.tar'; bt_drive = f'{M}/bytetrack_x_mot17.pth.tar'
    os.makedirs(os.path.dirname(bt_local), exist_ok=True)
    if not os.path.exists(bt_local):
        if os.path.exists(bt_drive):
            shutil.copy(bt_drive, bt_local); print('ByteTrack: from Drive')
        else:
            os.system('pip install -q -U gdown'); import gdown
            gdown.download(id='1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5', output=bt_local, quiet=False)
            if os.path.exists(bt_local): shutil.copy(bt_local, bt_drive)
    print('Models ready (ByteTrack fallback — no AIC25 training).')
else:
    print('Local: models in place.')

---
## Step 4 — Config: scene, cameras, frame cap
`MAXF` caps frames per camera for speed. Set `MAXF = 0` to use all 9000 frames (much slower).

In [ ]:
SCENE   = 'Warehouse_016'
DATASET = 'Val'                                            # Val | Test
CAMERAS = ['Camera', 'Camera_01', 'Camera_02', 'Camera_03']  # None = all 12
MAXF    = 1500                                             # frames/camera; 0 = all 9000

os.chdir(REPO)
print(f'Scene {SCENE} ({DATASET}) | cameras {CAMERAS or "ALL"} | cap {MAXF or "ALL"} frames')

---
## Step 5 — Download dataset (HuggingFace) — videos + ground_truth

In [ ]:
if ON_COLAB:
    import getpass
    from huggingface_hub import snapshot_download, login
    dd=f'{DRIVE}/datasets/{DATASET}/{SCENE}'; ld=f'{REPO}/AIC25_Track1/{DATASET}/{SCENE}'
    dv=f'{dd}/videos'; vl=f'{ld}/videos'
    os.makedirs(dd, exist_ok=True); os.makedirs(ld, exist_ok=True)
    if os.path.exists(dv) and os.listdir(dv):
        print('[CACHE HIT] videos on Drive.')
    else:
        tok=None
        try:
            from google.colab import userdata; tok=userdata.get('HF_TOKEN')
        except Exception: pass
        if not tok: tok=getpass.getpass('HF token: ')
        login(token=tok); sp=DATASET.lower()
        print(f'Downloading {SCENE} ({DATASET})...', flush=True)
        snapshot_download('nvidia/PhysicalAI-SmartSpaces', repo_type='dataset', local_dir='/content/hf_tmp',
            allow_patterns=[f'MTMC_Tracking_2025/{sp}/{SCENE}/videos/**',
                            f'MTMC_Tracking_2025/{sp}/{SCENE}/calibration.json',
                            f'MTMC_Tracking_2025/{sp}/{SCENE}/ground_truth.json'])
        src=f'/content/hf_tmp/MTMC_Tracking_2025/{sp}/{SCENE}'
        if not os.path.exists(dv): shutil.copytree(f'{src}/videos', dv)
        for fn in ['calibration.json','ground_truth.json']:
            if os.path.exists(f'{src}/{fn}'): shutil.copy(f'{src}/{fn}', f'{dd}/{fn}')
        shutil.rmtree('/content/hf_tmp', ignore_errors=True)
    for fn in ['calibration.json','ground_truth.json']:
        s,d=f'{dd}/{fn}',f'{ld}/{fn}'
        if os.path.exists(s) and not os.path.exists(d): shutil.copy(s,d)
    if os.path.islink(vl): os.unlink(vl)
    os.makedirs(vl, exist_ok=True); os.makedirs(f'{ld}/depth_map', exist_ok=True)
    cams=sorted(os.path.splitext(f)[0] for f in os.listdir(dv) if f.endswith('.mp4'))
    print(f'✓ {len(cams)} cameras available: {cams}')
else: print('Local: existing data.')

---
## Step 6 — Link outputs (Detection + Tracking → Drive; **EmbedFeature → LOCAL**)
EmbedFeature stays on local disk — writing its hundreds of thousands of tiny `.npy` files to Drive is ~100× slower.

In [ ]:
if ON_COLAB:
    for folder in ['Detection','Tracking']:          # persist to Drive
        df=f'{DRIVE}/outputs/{folder}'; rf=f'{REPO}/{folder}'; os.makedirs(df, exist_ok=True)
        if os.path.islink(rf): pass
        elif os.path.isdir(rf):
            for it in os.listdir(rf):
                s,d=f'{rf}/{it}',f'{df}/{it}'
                if not os.path.exists(d): shutil.move(s,d)
            shutil.rmtree(rf, ignore_errors=True); os.symlink(df, rf)
        else: os.symlink(df, rf)
        print(f'{folder}/ → Drive')
    ef=f'{REPO}/EmbedFeature'                          # LOCAL (fast)
    if os.path.islink(ef): os.unlink(ef)
    os.makedirs(ef, exist_ok=True)
    print('EmbedFeature/ → LOCAL', os.path.realpath(ef))
else: print('Local.')

---
## Generate single-camera JSONs (fresh, frame-capped)
Re-detects fresh each run (capped to `MAXF`) so a stale/truncated cached detection can't hang the embedding step. Order: detect → embed → track → fix.

In [ ]:
import subprocess
os.chdir(REPO)
_dv=f'{DRIVE}/datasets/{DATASET}/{SCENE}/videos' if DRIVE else None
_lv=f'{REPO}/AIC25_Track1/{DATASET}/{SCENE}/videos'
cam_source=_dv if (_dv and os.path.exists(_dv)) else _lv
all_cams=sorted(os.path.splitext(f)[0] for f in os.listdir(cam_source) if f.endswith('.mp4'))
cams=[c for c in CAMERAS if c in all_cams] if CAMERAS else all_cams
print('Cameras:', cams, '| cap', MAXF or 'ALL')

if os.path.exists(f'{REPO}/BoT-SORT/ai_city_ckpt.pth.tar'):
    ckpt='BoT-SORT/ai_city_ckpt.pth.tar'; exp='BoT-SORT/yolox/exps/example/mot/yolox_x_AI_City_25.py'
else:
    ckpt='BoT-SORT/pretrained/bytetrack_x_mot17.pth.tar'; exp='BoT-SORT/yolox/exps/example/mot/yolox_x_mix_det.py'
det_cap = f'--max_frames {MAXF}' if MAXF else ''
trk_cap = f'--limit_frames {MAXF}' if MAXF else ''

# fresh detections/embeddings (only our cams) so nothing stale hangs the run
shutil.rmtree(f'{REPO}/Detection/{SCENE}', ignore_errors=True);  os.makedirs(f'{REPO}/Detection/{SCENE}', exist_ok=True)
shutil.rmtree(f'{REPO}/EmbedFeature/{SCENE}', ignore_errors=True)

def run(cmd):
    r=subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.returncode!=0: print('\n'.join(r.stdout.strip().splitlines()[-30:]))
    return r.returncode

for cam in cams:
    print(f'\n=== detect {cam} ({MAXF or "all"} frames) ===')
    run(f'{PY} BoT-SORT/tools/aic25_get_detection.py --scene {SCENE} --dataset {DATASET} '
        f'--camera {cam} -f {exp} -c {ckpt} {det_cap} ./')

print('\n=== [D] embeddings (OSNet) ===')
os.chdir(f'{REPO}/deep-person-reid')
print('rc', os.system(f'{PY} torchreid/aic25_extract.py -s {SCENE} --dataset {DATASET} ../'))
os.chdir(REPO)

for cam in cams:
    print(f'\n=== track+fix {cam} ===')
    run(f'{PY} BoT-SORT/single_camera_tracking.py -s {SCENE} -c {cam} --dataset {DATASET} {trk_cap}')
    run(f'{PY} BoT-SORT/single_camera_fix.py -s {SCENE} -c {cam} --dataset {DATASET}')
print('\n[DONE] capped run complete — JSONs ready for the ablation cell.')

---
## Tier-1 — Tracklet Repair Ablation (your result)

In [ ]:
import json
os.chdir(REPO)
VARIANTS=['baseline','interpolation_only','merge_only','full_repair']
METRICS=['total_detections','num_tracklets','mean_tracklet_length',
         'num_tracklets_with_gaps','total_internal_gaps']
summary={}
for cam in cams:
    raw=f'Tracking/Singlecamera/{SCENE}/{cam}/{cam}.json'
    if not os.path.exists(raw): print(f'[skip] {cam}: no {raw}'); continue
    outdir=f'tracklet_repair/results/ablation_{SCENE}_{cam}'
    subprocess.run(f'{PY} -m tracklet_repair.src.evaluation.run_ablation --input-json {raw} '
                   f'--output-dir {outdir} --max-gap 5 --max-merge-gap 5 --max-center-distance 80 '
                   f'--max-size-ratio 1.5 --short-threshold 10', shell=True)
    try:
        abl=json.load(open(f'{outdir}/ablation.json'))
        summary[cam]={v:abl['variants'][v]['statistics'] for v in VARIANTS}
    except Exception as e:
        print(f'[warn] {cam}: {e}')

print(f'\n{"="*62}\nTIER-1 SUMMARY — baseline vs full_repair\n{"="*62}')
print(f'{"camera":<12}{"metric":<24}{"baseline":>10}{"full_repair":>12}')
print('-'*58)
for cam,vd in summary.items():
    for m in METRICS:
        b=vd['baseline'].get(m); f=vd['full_repair'].get(m)
        bs=f'{b:.2f}' if isinstance(b,float) else str(b)
        fs=f'{f:.2f}' if isinstance(f,float) else str(f)
        print(f'{cam:<12}{m:<24}{bs:>10}{fs:>12}')
    print()
print('Per-camera 4-way tables: tracklet_repair/results/ablation_%s_<cam>/ablation.md' % SCENE)

---
## Save results to Drive (before disconnect!)
Outputs live in `/content/repo` which wipes on disconnect — copy them to Drive.

In [ ]:
if ON_COLAB:
    shutil.copytree(f'{REPO}/tracklet_repair/results',
                    f'{DRIVE}/outputs/tracklet_repair_results', dirs_exist_ok=True)
    print('Saved ablation results to Drive ✓')
else:
    print('Local: results already in repo.')